# Tree Based Ensemble(XGBoost)

[Task]
1. 성과출력: Learning Curve, Lift Chart..  
2. Train 기준 Perfect Fitting 만들기
3. Bias와 Variance를 고려한 Best Fitting 만들기

[Data]  
Wine Quality Data set  
https://archive.ics.uci.edu/dataset/186/wine+quality

[Note]  
오류방지를 위한 최소한의 전처리만 수행

In [ ]:
#  UCI Machine Learning Data Load Library
!pip install ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from ucimlrepo import fetch_ucirepo


In [ ]:
## 1. 데이터 로드
wine_quality = fetch_ucirepo(id=186)
X = wine_quality.data.features
y = wine_quality.data.targets['quality']

# Concatenate
df = pd.concat([X, y], axis=1)

print(f"Data Shape: {df.shape}")
print(f"Categorical Variable: {X.select_dtypes(include=['object']).columns.tolist()}")

print(f"Info: {df.info()}")
print(f"Samples: {df.head(5)}")

In [ ]:
## 2. 전처리

# 범주형 변수 전처리 (더미 코딩)
X = pd.get_dummies(X, drop_first=True)

# 데이터 분할: (Train+Val) vs Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 데이터 분할 2차: Train vs Val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42
)

print(f"\n=== 데이터 분할 ===")
print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val:   {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")


In [ ]:
## 3. 모델링

# 하이퍼파라미터(Baseline)
params = {
    'n_estimators': 100,
    'max_depth': 3,
    'learning_rate': 0.1,
    'min_child_weight' : 10,
    'subsample': 1,
    'colsample_bytree': 1,
    'random_state': 42
}

# Task1. Train 기준 Perfect Fitting 을 만들어보세요.
params_overfitting = {
    'n_estimators': 100,
    'max_depth': 3,
    'learning_rate': 0.1,
    'min_child_weight' : 10,
    'subsample': 1,
    'colsample_bytree': 1,
    'random_state': 42
}

# Task2. Bias와 Variance를 고려한 Best Fitting 을 만들어보세요.
params_best = {
    'n_estimators': 100,
    'max_depth': 3,
    'learning_rate': 0.1,
    'min_child_weight' : 10,
    'subsample': 1,
    'colsample_bytree': 1,
    'random_state': 42
}

# 모델 학습 with selected params
selected_params = params
#selected_params = params_overfitting
#selected_params = params_best

model = XGBRegressor(**selected_params, eval_metric='mae')

model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

# 예측
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)


# 성능 평가
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)


print(f"\n=== 모델링 완료 ===")

In [ ]:
## 4. 성과분석

print(f"\n=== 모델 성능 ===")
print(f"Train MAE: {train_mae:.4f}, Train R²: {train_r2:.4f}")
print(f"Val MAE: {val_mae:.4f}, Train R²: {val_r2:.4f}")
print(f"Test MAE: {test_mae:.4f}, Test R²: {test_r2:.4f}")
print(f"Val-Train MAE 차이: {val_mae - train_mae:.4f}")
print(f"Test-Train MAE 차이: {test_mae - train_mae:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --------------------------------------------------------
# 4-1. Learning Curve
# --------------------------------------------------------
results = model.evals_result()
train_mae_all = results['validation_0']['mae']
val_mae_all = results['validation_1']['mae']

# 전체 epochs 중 n개 포인트만 샘플링
sample_n = 20
total_epochs = len(train_mae_all)
step = max(1, total_epochs // sample_n)
sample_indices = list(range(0, total_epochs, step))[:sample_n]
sample_indices.append(total_epochs - 1)  # 마지막 포인트 추가

# 러닝커브 출력
sampled_epochs = [i for i in sample_indices]
sampled_train_mae = [train_mae_all[i] for i in sample_indices]
sampled_val_mae = [val_mae_all[i] for i in sample_indices]

axes[0, 0].plot(sampled_epochs, sampled_train_mae, 'o-',
                label='Train MAE', color='blue', linewidth=2, markersize=8)
axes[0, 0].plot(sampled_epochs, sampled_val_mae, 'o-',
                label='Validation MAE', color='red', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Boosting Round')
axes[0, 0].set_ylabel('MAE')
axes[0, 0].set_title('Learning Curve (n_estimators)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# --------------------------------------------------------
# 4-2. Lift Chart - Train Set
# --------------------------------------------------------
df_train = pd.DataFrame({'actual': y_train, 'predicted': y_train_pred})
df_train['pred_bin'] = pd.qcut(df_train['predicted'], q=10, labels=False, duplicates='drop')
dist_train = df_train.groupby('pred_bin').agg({
    'actual': 'mean',
    'predicted': 'mean'
}).reset_index()

axes[0, 1].plot(dist_train['pred_bin'], dist_train['actual'], 'o-',
                label='Actual', color='blue', linewidth=2, markersize=8)
axes[0, 1].plot(dist_train['pred_bin'], dist_train['predicted'], 'o-',
                label='Predicted', color='red', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Prediction Decile')
axes[0, 1].set_ylabel('Average Quality')
axes[0, 1].set_title('Lift Chart - Train Set')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# --------------------------------------------------------
# 4-3. Lift Chart - Validation Set
# --------------------------------------------------------
df_val = pd.DataFrame({'actual': y_val, 'predicted': y_val_pred})
df_val['pred_bin'] = pd.qcut(df_val['predicted'], q=10, labels=False, duplicates='drop')
dist_val = df_val.groupby('pred_bin').agg({
    'actual': 'mean',
    'predicted': 'mean'
}).reset_index()

axes[1, 0].plot(dist_val['pred_bin'], dist_val['actual'], 'o-',
                label='Actual', color='blue', linewidth=2, markersize=8)
axes[1, 0].plot(dist_val['pred_bin'], dist_val['predicted'], 'o-',
                label='Predicted', color='red', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Prediction Decile')
axes[1, 0].set_ylabel('Average Quality')
axes[1, 0].set_title('Lift Chart - Validation Set')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# --------------------------------------------------------
# 4-4. Lift Chart - Test Set
# --------------------------------------------------------
df_test = pd.DataFrame({'actual': y_test, 'predicted': y_test_pred})
df_test['pred_bin'] = pd.qcut(df_test['predicted'], q=10, labels=False, duplicates='drop')
dist_test = df_test.groupby('pred_bin').agg({
    'actual': 'mean',
    'predicted': 'mean'
}).reset_index()

axes[1, 1].plot(dist_test['pred_bin'], dist_test['actual'], 'o-',
                label='Actual', color='blue', linewidth=2, markersize=8)
axes[1, 1].plot(dist_test['pred_bin'], dist_test['predicted'], 'o-',
                label='Predicted', color='red', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Prediction Decile')
axes[1, 1].set_ylabel('Average Quality')
axes[1, 1].set_title('Lift Chart - Test Set')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
## 5. Feature Importance

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n=== Feature Importance (Top 10) ===")
print(feature_importance.head(10).to_string(index=False))

# Feature Importance 시각화
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(10)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()